# NLP Preprocessing Assignment

This notebook covers fundamental NLP preprocessing techniques and includes a chunking task using the CoNLL-2000 dataset.

**Why CoNLL-2000?**  
The CoNLL-2000 dataset is specifically designed for chunking (identifying phrases like noun phrases and verb phrases). It's different from CoNLL-2003, which is used for Named Entity Recognition (NER). Using the right dataset for the task is important.

---

## Task 1: Conceptual Questions

### Q1. What is the difference between "Love" and "love" in NLP?

Most NLP systems are case-sensitive, so "Love" and "love" are treated as different tokens. This can mess up frequency counts and unnecessarily bloat the vocabulary. Lowercasing during preprocessing fixes this by treating both as the same word.

### Q2. What happens if stopwords are not removed?

Stopwords like "is", "the", "and" don't carry much meaning. If you don't remove them:
- They dominate word frequency counts
- They add noise to the model
- They make the dataset bigger than it needs to be

Removing them helps focus on the actual important words.

### Q3. When should you NOT remove stopwords?

**Sentiment analysis with negation:**  
"I am not happy" becomes "I am happy" if you remove "not". The meaning flips completely.

**Question answering or search:**  
In queries like "What is the capital of India?", words like "is" and "of" help preserve the structure. Removing them can hurt understanding.

### Q4. Difference between Stemming and Lemmatization

**Stemming:**  
Chops off word endings using basic rules. Fast but rough. Example: "studies" -> "studi" (not a real word).

**Lemmatization:**  
Converts words to their dictionary form using language rules. Slower but more accurate. Example: "studies" -> "study".

Use stemming when speed matters, lemmatization when accuracy matters.

---

## Setup: Install Libraries

In [ ]:
!pip install nltk -q

In [ ]:
import re
import string
from collections import Counter
import nltk

# Try to load NLTK stopwords, fall back to built-in list if needed
NLTK_AVAILABLE = False
try:
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt', quiet=True)
    nltk.download('conll2000', quiet=True)  # For chunking task
    from nltk.corpus import stopwords as nltk_stopwords
    _ = nltk_stopwords.words('english')
    NLTK_AVAILABLE = True
    print("NLTK loaded successfully.")
except Exception:
    NLTK_AVAILABLE = False
    print("NLTK unavailable - using fallback stopword list.")

# Fallback stopword list
BUILTIN_STOPWORDS = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you',
    "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself',
    'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her',
    'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them',
    'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom',
    'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was',
    'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do',
    'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or',
    'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with',
    'about', 'against', 'between', 'into', 'through', 'during', 'before',
    'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out',
    'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once',
    'here', 'there', 'when', 'where', 'why', 'how', 'all', 'both', 'each',
    'few', 'more', 'most', 'other', 'some', 'such', 'own', 'same', 'so',
    'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don',
    "don't", 'should', "should've", 'now', 'd', 'll', 'm', 'o', 're',
    've', 'y', 'ain', 'aren', "aren't", 'couldn', "couldn't", 'didn',
    "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't",
    'haven', "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't",
    'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn',
    "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't",
    'wouldn', "wouldn't"
}

def get_stopwords():
    if NLTK_AVAILABLE:
        return set(nltk_stopwords.words('english'))
    return BUILTIN_STOPWORDS

print("Setup complete.")

## Task 2: Text Preprocessing Function

In [ ]:
def preprocess_text(text):
    """
    Preprocess text for NLP tasks.
    
    Steps:
    - Handle edge cases (None, empty, numbers-only, emojis-only)
    - Remove URLs and emails
    - Remove emojis
    - Fix repeated characters (e.g., 'loooove' -> 'love')
    - Lowercase
    - Remove numbers and punctuation
    - Tokenize and remove short tokens
    - Remove stopwords (keep negations like 'not')
    
    Returns:
        tokens (list): Cleaned token list
        clean_sentence (str): Cleaned sentence string
    """
    
    # Handle None or non-string
    if text is None or not isinstance(text, str):
        return [], ""
    
    # Handle empty
    if text.strip() == "":
        return [], "[EMPTY]"
    
    # Handle numbers-only
    if re.fullmatch(r'[\d\s\+\-\.]+', text.strip()):
        return [], "[NUMBERS ONLY]"
    
    # Remove emojis and check if text remains
    emoji_pattern = r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\U00002702-\U000027B0\U000024C2-\U0001F251]+'
    test_text = re.sub(emoji_pattern, '', text).strip()
    if test_text == "":
        return [], "[EMOJIS ONLY]"
    
    # Remove URLs and emails
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    
    # Remove emojis
    text = re.sub(emoji_pattern, '', text)
    
    # Fix repeated characters (e.g., 'loooove' -> 'love')
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Lowercase
    text = text.lower()
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Clean up extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize
    tokens = text.split()
    
    # Keep meaningful short words like 'no', 'not'
    preserve = {'no', 'not', 'ok'}
    tokens = [t for t in tokens if len(t) > 2 or t in preserve]
    
    # Remove stopwords but keep negations
    stop_words = get_stopwords()
    negations = {'not', 'no', 'nor', 'neither', 'never', 'none'}
    stop_words = stop_words - negations
    
    tokens = [t for t in tokens if t not in stop_words]
    
    clean_sentence = ' '.join(tokens)
    
    return tokens, clean_sentence

# Quick test
sample = "WOW!!! This is GREAT!!!"
tokens, clean = preprocess_text(sample)
print(f"Input: {sample}")
print(f"Output: {clean}")

## Task 3: Test on Real Sentences

In [ ]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

results = []

print("Test Results:\n")
for i, sentence in enumerate(test_sentences, 1):
    tokens, clean = preprocess_text(sentence)
    results.append({'original': sentence, 'tokens': tokens, 'clean': clean})
    print(f"[{i}] Original: {sentence}")
    print(f"    Cleaned:  {clean}")
    print()

## Task 4: Token Statistics

In [ ]:
print("Token Analysis:\n")

for i, res in enumerate(results, 1):
    orig_count = len(res['original'].split())
    token_count = len(res['tokens'])
    unique_count = len(set(res['tokens']))
    avg_len = round(sum(len(t) for t in res['tokens']) / token_count, 2) if token_count > 0 else 0
    
    print(f"[{i}] Total tokens: {token_count}, Unique: {unique_count}, Avg length: {avg_len}")

# Find most noisy (most tokens removed)
noise_removed = [(i+1, len(r['original'].split()) - len(r['tokens']), r['original']) 
                 for i, r in enumerate(results)]
most_noisy = max(noise_removed, key=lambda x: x[1])

print(f"\nMost noisy sentence (#{most_noisy[0]}): \"{most_noisy[2]}\"")
print(f"Removed {most_noisy[1]} tokens as noise.")

## Task 5: Word Frequency Analysis

In [ ]:
# Combine all tokens
all_tokens = []
for res in results:
    all_tokens.extend(res['tokens'])

print(f"Total tokens: {len(all_tokens)}")
print(f"Unique tokens: {len(set(all_tokens))}\n")

freq = Counter(all_tokens)

print("Top 10 most frequent words:")
for word, count in freq.most_common(10):
    print(f"  {word}: {count}")

print("\nTop 5 least frequent words:")
for word, count in freq.most_common()[:-6:-1]:
    print(f"  {word}: {count}")

## Task 6: Full Pipeline Function

In [ ]:
def full_pipeline(text_list):
    """
    Process a list of sentences.
    
    Args:
        text_list (list): List of raw text strings
    
    Returns:
        dict: {'tokens': [...], 'clean_sentences': [...]}
    """
    if not isinstance(text_list, list):
        raise TypeError("Input must be a list.")
    
    if len(text_list) == 0:
        return {"tokens": [], "clean_sentences": []}
    
    all_tokens = []
    all_clean = []
    
    for text in text_list:
        tokens, clean = preprocess_text(text)
        all_tokens.append(tokens)
        all_clean.append(clean)
    
    return {"tokens": all_tokens, "clean_sentences": all_clean}

# Test pipeline
output = full_pipeline(test_sentences)
print("Pipeline output (first 3):")
for i in range(3):
    print(f"[{i+1}] {output['clean_sentences'][i]}")

## Task 7: Edge Case Testing

In [ ]:
edge_cases = [
    ("", "Empty string"),
    ("   ", "Only whitespace"),
    ("1234567890", "Only numbers"),
    (None, "None input"),
]

print("Edge Case Tests:\n")
for val, label in edge_cases:
    tokens, clean = preprocess_text(val)
    print(f"{label}:")
    print(f"  Input: {repr(val)}")
    print(f"  Output: {repr(clean)}")
    print()

## Task 8: Text Chunking with CoNLL-2000

**Dataset: CoNLL-2000**  
This dataset is specifically designed for chunking tasks. Chunking identifies syntactic phrases like noun phrases (NP) and verb phrases (VP) in sentences.

**Why CoNLL-2000 and not CoNLL-2003?**  
CoNLL-2003 is for Named Entity Recognition (identifying names, locations, organizations). CoNLL-2000 is for chunking. Using the wrong dataset would give bad results because the annotations are different.

Let's load the CoNLL-2000 dataset and extract some basic chunking patterns.

In [ ]:
try:
    from nltk.corpus import conll2000
    
    # Load train and test data
    train_sents = conll2000.chunked_sents('train.txt')
    test_sents = conll2000.chunked_sents('test.txt')
    
    print(f"CoNLL-2000 loaded successfully.")
    print(f"Training sentences: {len(train_sents)}")
    print(f"Test sentences: {len(test_sents)}")
    print(f"\nSample sentence with chunks:")
    print(train_sents[0])
    
    # Extract chunk types
    chunk_types = set()
    for sent in train_sents[:100]:  # Sample first 100 for speed
        for chunk in sent:
            if hasattr(chunk, 'label'):
                chunk_types.add(chunk.label())
    
    print(f"\nChunk types in dataset: {sorted(chunk_types)}")
    
except Exception as e:
    print(f"Could not load CoNLL-2000: {e}")
    print("Make sure you have internet connection for the first download.")

### Simple Chunking Example

Let's build a basic chunker using regex patterns to identify noun phrases.

In [ ]:
try:
    # Download POS tagger if needed
    nltk.download('averaged_perceptron_tagger', quiet=True)
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    
    # Define a simple grammar for noun phrase chunking
    # NP chunk: optional determiner + any number of adjectives + at least one noun
    grammar = r"""
        NP: {<DT>?<JJ.*>*<NN.*>+}
        VP: {<VB.*><NP|PP>}
    """
    
    chunker = nltk.RegexpParser(grammar)
    
    # Test on a sample sentence
    test_sentence = "The quick brown fox jumps over the lazy dog"
    words = nltk.word_tokenize(test_sentence)
    pos_tags = nltk.pos_tag(words)
    
    print(f"Sentence: {test_sentence}")
    print(f"\nPOS tags: {pos_tags}")
    
    # Apply chunking
    chunks = chunker.parse(pos_tags)
    print(f"\nChunked:")
    print(chunks)
    
    # Extract noun phrases
    print(f"\nExtracted noun phrases:")
    for subtree in chunks.subtrees():
        if subtree.label() == 'NP':
            print(" ".join([word for word, tag in subtree.leaves()]))
            
except Exception as e:
    print(f"Chunking failed: {e}")

### Evaluate Chunker on CoNLL-2000

Let's test our simple chunker on a few sentences from the CoNLL-2000 test set.

In [ ]:
try:
    # Test on first 5 sentences from CoNLL-2000
    print("Testing simple chunker on CoNLL-2000 data:\n")
    
    for i, sent in enumerate(test_sents[:5]):
        # Extract words and POS tags
        words_tags = [(word, pos) for word, pos, chunk in nltk.chunk.tree2conlltags(sent)]
        
        # Get our predictions
        predicted = chunker.parse(words_tags)
        
        print(f"Sentence {i+1}: {' '.join([w for w, t in words_tags])}")
        print(f"Gold chunks: {sent}")
        print(f"Predicted chunks: {predicted}")
        print("-" * 60)
        
except Exception as e:
    print(f"Evaluation failed: {e}")

---

## Summary

This notebook covered:
1. Basic NLP concepts (lowercasing, stopword removal, stemming vs lemmatization)
2. Text preprocessing pipeline (cleaning, tokenization, normalization)
3. Token frequency analysis
4. Edge case handling
5. **Chunking with CoNLL-2000 dataset** (the correct dataset for chunking tasks)

The CoNLL-2000 dataset is specifically designed for chunking (identifying syntactic phrases), while CoNLL-2003 is for Named Entity Recognition. Using the correct dataset matters for getting good results.